# Annotate samples 

Use `pyannotations` for best ease to annotate samples. Note, the samples here are larger than the intended ones and additionally marked, as easier to annotate in context + may want to make a judgement call whether to treat as labelled or keep for the unsupervised portion.

In [1]:
# re-loads code before cell execution, so if sth changes it will propagate:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys, shutil
import logging

import time as tm
import numpy as np
import pandas as pd

from PIL import Image
from ipyannotations import images

sys.path.append('..')

import src.utils as ssut
import src.data_utils as sdut
import src.geometry as sgut

## Set up annnotation

In [9]:
data_path = '../data/xView/to_annotate/'
df = pd.read_csv(f'{data_path}labels.csv')

out_path_labels = '../data/xView/labels_test/'
os.makedirs(out_path_labels, exist_ok= True)

to_annot_names = [ x for x in os.listdir( data_path) if not x.endswith('.csv')]
to_annot = [os.path.join( data_path, name) for name in to_annot_names]

In [10]:
def on_finish( df, label_di, labels):
    assert set(df['im_name'].values) == set(label_di.keys())
    # this assertion will fail if more clicks than there is images, and thus break the loop:
    assert len(df) == len(labels)
    # map labels to image names:
    df['labels'] = df['im_name'].map( label_di)
    df.loc[ df['labels'] == 'road', 'roads'] = 1
    df.loc[ df['labels'] == 'no road', 'roads'] = 0
    # save:
    name_split = df['im_name'].str.split('_').map(lambda x: x[0]).unique()
    assert len(name_split)==1
    which_tiff = name_split[0]
    out_to_ = f'{out_path_labels}for_tiff{which_tiff}.csv'
    print('------------------------------------------------------------------------------------------------------------')
    print(f'---------------------------  Outputting to: {out_to_}. --------------------')
    print('------------------------------------------------------------------------------------------------------------')
    df.to_csv( out_to_, index= False)

In [11]:
widget = images.ClassLabeller(
    options=['road','no road','unlabel','reject']
)
labels = []
label_di = {}

def annotate( annot):
    labels.append( annot)
    # to ensure correct matching of labels to file names:
    global next_ 
    label_di[ next_.split('/')[-1]] = annot
    try:
        next_ = to_annot.pop(0)
        widget.display( next_)
    except IndexError:
        print('------------------------------------------------------------------------------------------------------------')
        print('---------------------------- Aaaaand your task is done, ran out of images in the folder! -------------------')
        print('------------------------------------------------------------------------------------------------------------')
        on_finish( df, label_di, labels)
        
widget.on_submit( annotate)

## Annotate interactively


In [12]:
global next_ 
next_ = to_annot.pop(0)
widget.display( next_)
widget

ClassLabeller(children=(Box(children=(Output(layout=Layout(margin='auto', min_height='50px')),), layout=Layout…

------------------------------------------------------------------------------------------------------------
---------------------------- Aaaaand your task is done, ran out of images in the folder! -------------------
------------------------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------------------------
---------------------------  Outputting to ../data/xView/labels_test/for_tiff1175.csv. -----------
------------------------------------------------------------------------------------------------------------


### can sanity-check if wanting to...

In [14]:
label_di == dict(zip(to_annot_names,labels))

True

In [13]:
len(labels), len(to_annot_names)

(5, 5)

In [16]:
df.head(3)

,xs,ys,buildings,roads,cars,bbox_np,image,bbox_context,bbox_in_context,built_area,n_cars,n_bus_trucks,im_name,labels
0,219,1758,-1,-1,-1,"[155, 1694, 283, 1822]",tif1175_x219_y1758.png,"[91, 1630, 347, 1886]","[64, 64, 192, 192]",0.927246,0.0,0.0,1175_0.png,reject
1,452,2275,-1,-1,-1,"[388, 2211, 516, 2339]",tif1175_x452_y2275.png,"[324, 2147, 580, 2403]","[64, 64, 192, 192]",1.247437,0.0,0.0,1175_1.png,reject
2,886,2113,-1,-1,-1,"[822, 2049, 950, 2177]",tif1175_x886_y2113.png,"[758, 1985, 1014, 2241]","[64, 64, 192, 192]",0.748962,0.0,0.0,1175_2.png,reject
